# Data Cleaning Strategy

| Issue                                                                    | Treatment                                                 | Justification                                                             |
| ------------------------------------------------------------------------ | --------------------------------------------------------- | ------------------------------------------------------------------------- |
| 5,268 duplicate rows                                                     | Remove                                                    | Exact duplicates across all columns; redundant records.                   |
| InvoiceDate stored as object                                             | Convert to datetime                                       | Incorrect datatype for date analysis.                                     |
| 1,454 missing Description values                                         | Impute 1,033 values using StockCode mapping               | One-to-one StockCode → Description relationship confirmed.                |
| Remaining 421 missing Description values                                 | Retain as missing                                         | Cannot be safely recovered without assumptions.                           |
| StockCode-Description inconsistencies (e.g., StockCode 22139 → "amazon") | Correct using dominant description for that StockCode     | Likely data entry inconsistency.                                          |
| 127 operational descriptions ("check", "damages", "broken", etc.)        | Standardize into an "Operational Record" category or flag | These are not products and may require separate analysis.                 |
| CustomerID missing values                                                | Retain as missing                                         | Missingness appears to be a business characteristic rather than an error. |
| Country = "Unspecified"                                                  | Keep as-is                                                | Represents unknown customer location.                                     |
| Country = "European Community"                                           | Keep as-is                                                | Business/geographic classification, not an error.                         |
| InvoiceNo beginning with "C"                                             | Keep                                                      | Valid cancellation/return transactions.                                   |
| Negative Quantity values                                                 | Keep                                                      | Represent returns/cancellations.                                          |
| Negative UnitPrice values                                                | Keep                                                      | Represent bad debt adjustments.                                           |
| UnitPrice = 0 records                                                    | Keep                                                      | Includes both operational records and legitimate free transactions.       |


## Objective

The objective of this notebook is to apply the cleaning decisions identified during the assessment phase and prepare a high-quality dataset for analysis.

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/content/OnlineRetail.csv', encoding='latin1')

###Removing Duplicate rows

In [4]:
df.drop_duplicates(inplace=True)

###Converting `InvoiceDate` column to datetime data type

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

###Imputing missing values of `Description` column
During the data quality assessment, 1,033 missing Description values were identified as potentially recoverable using StockCode–Description mappings. Further investigation revealed that one StockCode (84611B) mapped to an operational description ("thrown away") rather than a product name. To avoid introducing incorrect product information, this StockCode was excluded from the imputation process. The remaining StockCodes were considered safe for description imputation due to their one-to-one relationship with valid product descriptions.

In [6]:
df['Description'].isnull().sum()

np.int64(1454)

In [7]:
sc_with_nan_desc = df[df['Description'].isnull()]['StockCode']

In [8]:
temp = df.groupby('StockCode')['Description'].nunique()
new_df = temp[temp==1]
sc_with_one_desc = df[df['StockCode'].isin(new_df.index) & ~(df['Description'].isnull())]
sc_with_one_desc =  sc_with_one_desc[['StockCode', 'Description']]
sc_with_one_desc.drop_duplicates(inplace=True)

In [9]:
desc_map = sc_with_one_desc.set_index('StockCode')['Description'].to_dict()

In [10]:
mask = df['Description'].isnull() & df['StockCode'].isin(desc_map)
df.loc[mask, 'Description'] = df.loc[mask, 'StockCode'].map(desc_map)

###Identification and Classification of Operational Records

In [11]:
df['RecordType'] = 'Product'
mask = (df['Description'].str.islower().fillna(False)) | (df['Description']=='?')
df.loc[mask, 'RecordType'] = 'Operational'

/tmp/ipykernel_1674/1691383562.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = (df['Description'].str.islower().fillna(False)) | (df['Description']=='?')


#Data Type Conversion

In [12]:
df['Country'] = df['Country'].astype('category')

In [13]:
df['RecordType'] = df['RecordType'].astype('category')

In [14]:
df['Description'] = df['Description'].astype('category')

In [15]:
df['CustomerID'] = df['CustomerID'].astype('Int64')

#Data Cleaning Summary
- Removed 5,268 duplicate records.
- Converted InvoiceDate to datetime format.
- Imputed recoverable missing Description values using StockCode mappings.
- Retained 421 unrecoverable Description values as missing.
- Created a RecordType column to distinguish Product and Operational records.
- Investigated StockCode–Description inconsistencies and determined that no further corrective action was required.
- Retained valid business records such as returns, cancellations, gift vouchers, bad debt adjustments, and transactions with missing CustomerID values.

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  object        
 1   StockCode    536641 non-null  object        
 2   Description  536220 non-null  category      
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[ns]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  Int64         
 7   Country      536641 non-null  category      
 8   RecordType   536641 non-null  category      
dtypes: Int64(1), category(3), datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 31.4+ MB


In [17]:
df.to_csv('online_retail_cleaned.csv', index=False)